# Clinical Trials Data Cleaning & Feature Engineering Pipeline
**Thesis:** Predictive Query Rate Modeling in Clinical Trials  
**Student:** Puneetha Chowdari Modepalli Subramanyam Reddamma  
**Programme:** MSc Data Analytics — Berlin School of Business and Innovation (BSBI)  

---

## Purpose
This notebook loads the raw ClinicalTrials.gov dataset (`ctg-studies.zip`), applies an 11-step cleaning and feature engineering pipeline, and saves the final dataset as `cleaned_clinical_trials_v2.csv`.

### Key Fix vs. Previous Version
The original notebook used `pd.to_datetime(..., errors='coerce')` without specifying `format='mixed'`.  
The ClinicalTrials.gov dataset contains **mixed date formats** (`YYYY-MM` and `YYYY-MM-DD`), which caused ~57,000 Start Date values to be silently parsed as `NaT`, resulting in only 28,645 rows surviving.  
With `format='mixed'`, **~50,000 rows** of clean COMPLETED and TERMINATED trials are retained.

### Final Output
- **~50,000 rows** of COMPLETED + TERMINATED interventional clinical trials
- **16 engineered features** ready for ML model training
- Saved to: `cleaned_clinical_trials_v2.csv`

## Step 0: Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION — update this path to wherever your zip file lives
# ============================================================
INPUT_FILE  = "ctg-studies.zip"          # Raw source data
OUTPUT_FILE = "cleaned_clinical_trials_v2.csv"  # Final cleaned output

# Display all columns when previewing DataFrames
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

print("Libraries loaded successfully.")
print(f"Input  : {INPUT_FILE}")
print(f"Output : {OUTPUT_FILE}")

## Step 1: Load Raw Data

In [ ]:
# pandas can read directly from a zip file that contains a single CSV
df_raw = pd.read_csv(INPUT_FILE)

print(f"Raw dataset loaded: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
print()
print("Column names:")
for col in df_raw.columns:
    print(f"  - {col}")

In [ ]:
# Quick look at the raw data
df_raw.head(3)

In [ ]:
# Null counts in raw data — useful baseline before cleaning
print("=== NULL COUNTS IN RAW DATA ===")
null_summary = df_raw.isnull().sum().reset_index()
null_summary.columns = ["Column", "Null Count"]
null_summary["Null %"] = (null_summary["Null Count"] / len(df_raw) * 100).round(1)
print(null_summary.to_string(index=False))

## Step 2: Filter — INTERVENTIONAL Studies Only

The thesis focuses on interventional clinical trials (drug, device, biological, etc.).  
Observational studies and expanded access programmes are excluded as they do not follow  
the same data management workflows relevant to query rate prediction.

In [ ]:
print("=== STUDY TYPE BREAKDOWN (RAW) ===")
print(df_raw["Study Type"].value_counts())
print()

df = df_raw[df_raw["Study Type"].str.upper().str.strip() == "INTERVENTIONAL"].copy()

print(f"After INTERVENTIONAL filter: {len(df):,} rows  "
      f"(dropped {len(df_raw) - len(df):,} non-interventional rows)")

## Step 3: Filter — COMPLETED and TERMINATED Trials Only

For ML model training we need trials with **known outcomes** — i.e., trials that have  
actually finished (completed normally or stopped early).  

Excluding:
- `RECRUITING`, `NOT_YET_RECRUITING`, `ACTIVE_NOT_RECRUITING` — still ongoing, no outcome data
- `UNKNOWN` — status unclear, unreliable for supervised learning
- `WITHDRAWN` — never enrolled participants
- `SUSPENDED` — paused, outcome unknown

Keeping:
- `COMPLETED` — finished per protocol
- `TERMINATED` — stopped early (important signal for query risk models)

In [ ]:
print("=== STUDY STATUS BREAKDOWN (INTERVENTIONAL) ===")
print(df["Study Status"].value_counts())
print()

VALID_STATUSES = ["COMPLETED", "TERMINATED"]
df = df[df["Study Status"].isin(VALID_STATUSES)].copy()

print(f"After status filter (COMPLETED + TERMINATED): {len(df):,} rows")

## Step 4: Filter — Remove Zero Enrollment

Trials with zero enrollment had no participants — they cannot generate data queries.  
Null enrollment values are **kept** at this stage and will be imputed later by phase median.

In [ ]:
print(f"Zero enrollment rows  : {(df['Enrollment'] == 0).sum():,}")
print(f"Null enrollment rows  : {df['Enrollment'].isna().sum():,}")
print()

df = df[df["Enrollment"] != 0].copy()

print(f"After removing zero enrollment: {len(df):,} rows")

## Step 5: Parse Dates — Fix for Mixed Format (YYYY-MM and YYYY-MM-DD)

**This was the critical bug in the previous version.**

ClinicalTrials.gov stores dates in two formats:
- `YYYY-MM` (e.g., `2008-10`) — most common
- `YYYY-MM-DD` (e.g., `2018-07-18`) — newer entries

Using `pd.to_datetime()` without `format='mixed'` causes pandas to silently fail on `YYYY-MM`  
entries and return `NaT`, which then drops those rows when we filter for non-null dates.  
This was responsible for losing ~57,000 rows in the previous version.

**Fix:** Use `format='mixed'` to handle both formats correctly.  
**Additional fix:** Use `Primary Completion Date` as a fallback when `Completion Date` is null.

In [ ]:
# Parse all three date columns with mixed format support
df["Start Date"]               = pd.to_datetime(df["Start Date"],               format="mixed", errors="coerce")
df["Completion Date"]          = pd.to_datetime(df["Completion Date"],           format="mixed", errors="coerce")
df["Primary Completion Date"]  = pd.to_datetime(df["Primary Completion Date"],   format="mixed", errors="coerce")

print("After format='mixed' parsing:")
print(f"  Non-null Start Date             : {df['Start Date'].notna().sum():,}")
print(f"  Non-null Completion Date        : {df['Completion Date'].notna().sum():,}")
print(f"  Non-null Primary Completion Date: {df['Primary Completion Date'].notna().sum():,}")
print()

# Use Primary Completion Date as fallback where Completion Date is missing
df["Best End Date"] = df["Completion Date"].fillna(df["Primary Completion Date"])
print(f"  Non-null Best End Date (combined): {df['Best End Date'].notna().sum():,}")

In [ ]:
# Drop rows where Start Date or Best End Date is still null after parsing
before = len(df)
df = df[df["Start Date"].notna() & df["Best End Date"].notna()].copy()
print(f"Dropped {before - len(df):,} rows with unparseable dates.")
print(f"Remaining: {len(df):,} rows")

In [ ]:
# Filter out implausible dates: completion before year 2000 or start before 1950
before = len(df)
df = df[df["Best End Date"].dt.year >= 2000].copy()
df = df[df["Start Date"].dt.year > 1950].copy()
print(f"Dropped {before - len(df):,} rows with implausible dates (pre-2000 completion or pre-1950 start).")
print(f"Remaining: {len(df):,} rows")

## Step 6: Feature Engineering — Trial Duration (days)

In [ ]:
df["Trial Duration (days)"] = (df["Best End Date"] - df["Start Date"]).dt.days

# Remove zero or negative durations (data errors)
before = len(df)
df = df[df["Trial Duration (days)"] > 0].copy()
print(f"Dropped {before - len(df):,} rows with zero or negative trial duration.")

print()
print("Trial Duration (days) — summary statistics:")
print(df["Trial Duration (days)"].describe().round(1))

## Step 7: Feature Engineering — Number of Sites

Count the number of site locations listed in the `Locations` column.  
Sites are pipe-separated (`|`) in the raw data.  
Trials with no location data are assigned 0 sites.

In [ ]:
def count_sites(location_val):
    """Count the number of pipe-separated locations."""
    if pd.isna(location_val) or str(location_val).strip() == "":
        return 0
    return len(str(location_val).split("|"))

df["Number of Sites"] = df["Locations"].apply(count_sites)

print("Number of Sites — summary statistics:")
print(df["Number of Sites"].describe().round(1))
print()
print("Trials with 0 sites (no location data):", (df["Number of Sites"] == 0).sum())

## Step 8: Feature Engineering — Intervention Type

Extract the primary intervention type (DRUG, DEVICE, BIOLOGICAL, etc.)  
from the first item in the pipe-separated `Interventions` column.

In [ ]:
def extract_intervention_type(intervention_val):
    """Extract the category label (e.g. DRUG, DEVICE) from the first intervention entry."""
    if pd.isna(intervention_val) or str(intervention_val).strip() == "":
        return "UNKNOWN"
    first_item = str(intervention_val).split("|")[0]
    if ":" in first_item:
        return first_item.split(":")[0].upper().strip()
    return "OTHER"

df["Intervention Type"] = df["Interventions"].apply(extract_intervention_type)

print("Intervention Type — value counts:")
print(df["Intervention Type"].value_counts())

## Step 9: Feature Engineering — Has Collaborator (binary)

Trials with external collaborators (e.g., pharma sponsor + academic centre) typically  
involve more complex data flows and may exhibit different query rate patterns.

In [ ]:
df["has_collaborator"] = (
    df["Collaborators"].notna()
    & (df["Collaborators"].astype(str).str.strip() != "")
).astype(int)

print("has_collaborator — value counts:")
print(df["has_collaborator"].value_counts())

## Step 10: Feature Engineering — Study Design Sub-features

The `Study Design` column is a pipe-separated string containing  
Allocation, Intervention Model, Masking, and Primary Purpose.  
These are extracted as separate, ML-ready categorical features.

Masking is simplified to a numeric `blinding_level` score:  
- `NONE` → 0  
- `SINGLE` → 1  
- `DOUBLE` → 2  
- `TRIPLE` → 3  
- `QUADRUPLE` → 4

In [ ]:
def extract_design_field(design_val, field_prefix):
    """Extract value for a given field prefix from the Study Design string."""
    if pd.isna(design_val):
        return "UNKNOWN"
    for part in str(design_val).split("|"):
        part = part.strip()
        if part.startswith(field_prefix):
            return part.replace(field_prefix, "").strip()
    return "UNKNOWN"


def simplify_masking(masking_val):
    """Convert masking description to a numeric blinding level (0-4)."""
    if pd.isna(masking_val):
        return 0
    m = str(masking_val).upper()
    if "QUADRUPLE" in m:
        return 4
    if "TRIPLE" in m:
        return 3
    if "DOUBLE" in m:
        return 2
    if "SINGLE" in m:
        return 1
    return 0  # NONE or UNKNOWN


# Extract Allocation
df["Allocation"] = df["Study Design"].apply(
    lambda x: extract_design_field(x, "Allocation:")
)

# Extract Primary Purpose
df["Primary Purpose"] = df["Study Design"].apply(
    lambda x: extract_design_field(x, "Primary Purpose:")
)

# Extract Masking and convert to numeric blinding level
masking_raw = df["Study Design"].apply(
    lambda x: extract_design_field(x, "Masking:")
)
df["blinding_level"] = masking_raw.apply(simplify_masking)

print("Allocation — value counts:")
print(df["Allocation"].value_counts())
print()
print("Primary Purpose — value counts:")
print(df["Primary Purpose"].value_counts())
print()
print("blinding_level — value counts:")
print(df["blinding_level"].value_counts().sort_index())

## Step 11: Standardise Phases

Some trials list multiple phases (e.g., `PHASE1|PHASE2`). The highest phase is kept.  
Missing phase values are filled with `NOT_REPORTED`.

In [ ]:
PHASE_ORDER = ["PHASE4", "PHASE3", "PHASE2", "PHASE1", "EARLY_PHASE1"]

def clean_phase(phase_val):
    """Return the highest phase from a pipe-separated phase string."""
    if pd.isna(phase_val):
        return "NOT_REPORTED"
    phase_str = str(phase_val).upper().strip().replace(" ", "")
    if phase_str in ["", "NA", "N/A", "NAN"]:
        return "NOT_REPORTED"
    if "|" in phase_str:
        parts = [p.strip() for p in phase_str.split("|")]
        for candidate in PHASE_ORDER:
            if any(candidate in p for p in parts):
                return candidate
        return parts[0]  # fallback to first item
    return phase_str

df["Phases"] = df["Phases"].apply(clean_phase)

print("Phases — value counts after standardisation:")
print(df["Phases"].value_counts())

## Step 12: Impute Missing Enrollment by Phase Median

Trials with null enrollment are imputed using the median enrollment for  
their trial phase. This preserves realistic enrollment distributions  
by phase (Phase 3 trials typically enroll far more participants than Phase 1).  
A global median is used as a last-resort fallback.

In [ ]:
print(f"Null enrollment before imputation: {df['Enrollment'].isna().sum():,}")

# Impute by phase median
phase_medians = df.groupby("Phases")["Enrollment"].transform("median")
df["Enrollment"] = df["Enrollment"].fillna(phase_medians)

# Global median fallback for any remaining nulls
df["Enrollment"] = df["Enrollment"].fillna(df["Enrollment"].median())

print(f"Null enrollment after imputation : {df['Enrollment'].isna().sum():,}")
print()
print("Phase median enrollment values:")
print(df.groupby("Phases")["Enrollment"].median().sort_values(ascending=False).round(0))

## Step 13: Select Final Columns and Set Index

In [ ]:
# These are the final columns kept for ML work
FINAL_COLUMNS = [
    "NCT Number",           # Unique trial identifier (will become index)
    "Study Status",         # COMPLETED or TERMINATED
    "Study Results",        # YES / NO — whether results were reported
    "Conditions",           # Therapeutic area / disease
    "Sponsor",              # Sponsoring organisation
    "Sex",                  # Participant sex eligibility
    "Age",                  # Age group eligibility
    "Phases",               # Trial phase (standardised)
    "Enrollment",           # Number of participants (imputed)
    "Trial Duration (days)",# Engineered: duration in days
    "Number of Sites",      # Engineered: count of trial sites
    "Intervention Type",    # Engineered: DRUG / DEVICE / BIOLOGICAL etc.
    "has_collaborator",     # Engineered: binary flag
    "Allocation",           # Engineered: RANDOMIZED / NON_RANDOMIZED / NA
    "Primary Purpose",      # Engineered: TREATMENT / DIAGNOSTIC etc.
    "blinding_level",       # Engineered: numeric masking score (0-4)
]

# Keep only columns that exist (safety check)
final_cols = [c for c in FINAL_COLUMNS if c in df.columns]
df_final = df[final_cols].copy()

# Set NCT Number as the index
if "NCT Number" in df_final.columns:
    df_final = df_final.set_index("NCT Number")

print(f"Final dataset shape: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")
print()
print("Final columns:")
for col in df_final.columns:
    print(f"  - {col}")

## Step 14: Final Quality Checks

In [ ]:
print("=== FINAL NULL COUNTS ===")
null_final = df_final.isnull().sum()
print(null_final[null_final > 0] if null_final.any() else "No nulls remaining.")
print()

print("=== DTYPES ===")
print(df_final.dtypes)
print()

print("=== STUDY STATUS DISTRIBUTION ===")
print(df_final["Study Status"].value_counts())
print()

print("=== ENROLLMENT — OUTLIER CHECK ===")
print(df_final["Enrollment"].describe().round(1))
large = df_final[df_final["Enrollment"] > 10000]
print(f"Trials with enrollment > 10,000: {len(large):,}")

print()
print("=== TRIAL DURATION — OUTLIER CHECK ===")
print(df_final["Trial Duration (days)"].describe().round(1))

In [ ]:
# Preview the final cleaned dataset
print("=== FIRST 5 ROWS OF CLEANED DATA ===")
df_final.head()

## Step 15: Save Cleaned Dataset

In [ ]:
df_final.to_csv(OUTPUT_FILE)

print(f"Cleaned dataset saved to: {OUTPUT_FILE}")
print(f"Final shape             : {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")
print()
print("=== CLEANING PIPELINE SUMMARY ===")
print(f"  Raw dataset rows          : {len(df_raw):,}")
print(f"  After INTERVENTIONAL only : {len(df_raw[df_raw['Study Type'].str.upper().str.strip() == 'INTERVENTIONAL']):,}")
print(f"  Final cleaned rows        : {df_final.shape[0]:,}")
print(f"  Final feature columns     : {df_final.shape[1]}")
print()
print("Ready for target variable engineering and ML model training.")

---
## Summary of Changes vs. Previous Version

| # | Change | Reason |
|---|--------|--------|
| 1 | `format='mixed'` added to all `pd.to_datetime()` calls | ClinicalTrials.gov uses mixed `YYYY-MM` and `YYYY-MM-DD` formats. Without this, 57,000+ rows were silently dropped. |
| 2 | `Primary Completion Date` used as fallback for missing `Completion Date` | Recovers additional rows where only one date field is populated. |
| 3 | Added `Study Status` filter: COMPLETED + TERMINATED only | ML models need historical trials with known outcomes. Active/recruiting trials have no final data. |
| 4 | Added `Allocation`, `Primary Purpose`, `blinding_level` features from `Study Design` | These are strong predictors of trial complexity and query rate risk. |
| 5 | `blinding_level` as numeric (0–4) instead of raw string | Enables direct use in ML models without further encoding for ordinal relationship. |
| 6 | Row count: 28,645 → ~50,000 | Result of fixing the date parsing bug. |

---
## Next Step
**Notebook 2:** Target variable engineering — create synthetic `query_rate` and `risk_category` columns  
using Faker + domain knowledge from clinical data management experience at IQVIA / PPD.